DMP PDF
  ↓
1. pdfplumber extraction
  ↓
2. rule-based structure detection
  ↓
3. build narrative JSON
  ↓
4. save final JSON


# Part 1 — Imports

In [17]:
from pathlib import Path
import pandas as pd

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json

# Part 2 — Paths

In [18]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample7.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_json_path = project_root / "data" / "pdfplumber_blocks" / f"{pdf_path.stem}.json"
csv_output_path = project_root / "outputs" / "debug" / f"{pdf_path.stem}_structured_lines.csv"
final_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_pdfplumber.json"

print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

PDF exists: True
Skeleton exists: True


# Part 3 — Run pdfplumber extraction

In [19]:
blocks = save_pdfplumber_outputs(pdf_path)

print("Extracted lines:", len(blocks))
print("Saved pdfplumber JSON:", pdfplumber_json_path.exists())

[2026-05-13 11:47:33] Extracting line-level text with pdfplumber: sample7.pdf
[2026-05-13 11:47:33] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample7.json
[2026-05-13 11:47:33] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample7.txt
Extracted lines: 17
Saved pdfplumber JSON: True


# Part 4 — Run rule-based structure detection

In [20]:

structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)
print("Detected format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].head(100)

Detected format: unknown
label
content    16
section     1
Name: count, dtype: int64


,page,line_order,text,avg_font_size,is_bold,label,document_format
0,1,1,Resource/Data Sharing Plan,11.04,True,section,unknown
1,1,2,The study investigators are committed to the o...,11.04,False,content,unknown
2,1,3,will include demographic and clinical assessme...,11.04,False,content,unknown
3,1,4,behavioral tasks collected from human subjects...,11.04,False,content,unknown
4,1,5,Archive (NDA) Data Sharing Terms and Condition...,11.04,False,content,unknown
5,1,6,data and analyzed data at the item and subject...,11.04,False,content,unknown
6,1,7,according to the expected timeline (semi-annua...,11.04,False,content,unknown
7,1,8,whichever occurs first). The investigators wil...,11.04,False,content,unknown
8,1,9,"allow for the data sharing through NDA, and co...",11.04,False,content,unknown
9,1,10,sharing. The investigators will certify the qu...,11.04,False,content,unknown


# Part 5 — Inspect extracted lines

In [21]:
print("Detected document format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

Detected document format: unknown
label
content    16
section     1
Name: count, dtype: int64


# Part 6 — Save CSV debug file

In [22]:
csv_output_path.parent.mkdir(parents=True, exist_ok=True)

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].to_csv(csv_output_path, index=False, encoding="utf-8")

print("Saved CSV:", csv_output_path)

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample7_structured_lines.csv


# Part 7 — Build narrative JSON

In [23]:
final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

sections = final_json["narrative"]["template"]["section"]

print("Saved JSON:", final_json_path)
print("Number of sections:", len(sections))

for sec in sections:
    print(sec["order"], sec["title"], "| questions:", len(sec["question"]))

[2026-05-13 11:47:33] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample7_pdfplumber.json
Saved JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample7_pdfplumber.json
Number of sections: 1
1 Resource/Data Sharing Plan | questions: 0


# Part 8 — Inspect one section

In [24]:
sections[0]

{'id': 'section_1',
 'title': 'Resource/Data Sharing Plan',
 'description': 'The study investigators are committed to the open and timely dissemination of study data. The final datasets\nwill include demographic and clinical assessment data obtained from interviews, questionnaires, and\nbehavioral tasks collected from human subjects. The investigators will be fully compliant to the NIMH Data\nArchive (NDA) Data Sharing Terms and Conditions, including submitting and harmonizing all descriptive/raw\ndata and analyzed data at the item and subject-level to the National Database for Clinical Trials (NDCT)\naccording to the expected timeline (semi-annually, at the end of the grant or at the time of publication,\nwhichever occurs first). The investigators will include appropriate language in participant consent documents to\nallow for the data sharing through NDA, and complete all required paperwork related to the submission and\nsharing. The investigators will certify the quality of all data